## **03_ml_00_2026s1: ML0 (RandomForest Classification)**
**Safe to eat or deadly poison?**

This homework is a classification task to identify whether a mushroom is edible or poisonous.

This dataset includes descriptions of hypothetical samples corresponding to 23 species of gilled mushrooms in the Agaricus and Lepiota Family Mushroom drawn from The Audubon Society Field Guide to North American Mushrooms (1981).

Each species is identified as definitely edible, definitely poisonous, or of unknown edibility and not recommended. This latter class was combined with the poisonous one. The Guide clearly states that there is no simple rule for determining the credibility of a mushroom; no rule like "leaflets three, let it be'' for Poisonous Oak and Ivy.


Step 1. Load 'mushroom2020_dataset.csv' data from the “Attachment” (note: this data set has been preliminarily prepared.).

Step 2. Drop rows where the target (label) variable is missing.

Step 3. Drop the following variables:
'id','gill-attachment', 'gill-spacing', 'gill-size','gill-color-rate', 'stalk-root', 'stalk-surface-above-ring', 'stalk-surface-below-ring', 'stalk-color-above-ring-rate','stalk-color-below-ring-rate','veil-color-rate','veil-type'

Step 4. Examine the number of rows, the number of digits, and whether any are missing.

Step 5. Fill missing values by adding the mean for numeric variables and the mode for nominal variables.

Step 6. Convert the label variable e (edible) to 1 and p (poisonous) to 0 and check the quantity. class0: class1

Step 7. Convert the nominal variable to numeric using a dummy code with drop_first = True.

Step 8. Split train/test with 20% test, stratify, and seed = 2020.

Step 9. Create a Random Forest on training data using the following parameters:
-	'criterion': ‘gini’
-	'max_depth: 3
-	'min_samples_leaf': 2
-	'n_estimators': 100
-	'random_state': 2020

Step 10.  Predict the testing data set with classification_report.


**Complete class MushroomClassifier from given code template below.**

In [2]:
#import your other libraries here
import pandas as pd
# hint
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.ensemble import RandomForestClassifier

In [ ]:
class MushroomClassifier:
    def __init__(self, data_path): # DO NOT modify this line
        self.data_path = data_path
        self.df = pd.read_csv(data_path)
        self.X_train = list()
        self.X_test = list()
        self.y_train = list()
        self.y_test = list()
        self.rf = None  # Your Random Forest Classfier

    def Q1(self): # DO NOT modify this line
        """
            1. (From step 1) Before doing the data prep., how many "na" are there in "gill-size" variables?
        """
        
        return self.df['gill-size'].isnull().sum()


    def Q2(self): # DO NOT modify this line
        """
            2. (From step 2-4) How many rows of data, how many variables?
            - Drop rows where the target (label) variable is missing.
            - Drop the following variables:
            'id','gill-attachment', 'gill-spacing', 'gill-size','gill-color-rate','stalk-root', 'stalk-surface-above-ring',
            'stalk-surface-below-ring', 'stalk-color-above-ring-rate','stalk-color-below-ring-rate','veil-color-rate','veil-type'
            - Examine the number of rows, the number of digits, and whether any are missing.
        """
        df_clean = self.df.dropna(subset=['label']).copy()
        df_clean.dropna(subset=['label'])
        df_clean = df_clean.drop(errors='ignore', columns=['id','gill-attachment', 'gill-spacing', 'gill-size','gill-color-rate','stalk-root', 'stalk-surface-above-ring',
                    'stalk-surface-below-ring', 'stalk-color-above-ring-rate','stalk-color-below-ring-rate','veil-color-rate','veil-type'])
        self.df = df_clean
        return self.df.shape


    def Q3(self):  # DO NOT modify this line
        self.Q2()  # remove this if you want to skip dependency and duplicate code
        
        num_cols = self.df.select_dtypes(include='number').columns.tolist()
        for col in num_cols:
            self.df[col] = self.df[col].fillna(self.df[col].mean())
            
        cat_cols = [col for col in self.df.columns if col not in num_cols and col != 'label']
        for col in cat_cols:
            self.df[col] = self.df[col].fillna(self.df[col].mode()[0])
            
        temp_labels = self.df['label'].map({'e': 1, 'p': 0})
        class0_quan = (temp_labels == 0).sum()
        class1_quan = (temp_labels == 1).sum()
        
        return (int(class0_quan), int(class1_quan))


    def Q4(self):  # DO NOT modify this Line
        self.Q3()  # remove this if you want to skip dependency and duplicate code
        
        X_raw = self.df.drop(columns=['label'])
        y = self.df['label'].map({'e': 1, 'p': 0})
        
        X_encoded = pd.get_dummies(X_raw, drop_first=True)
        
        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
            X_encoded, y, test_size=0.20, stratify=y, random_state=2020
        )
        
        return (self.X_train.shape, self.X_test.shape)


    def Q5(self):
        """
            5. (From step 9) 2nd best feature name based on the importance.
            Create a Random Forest on training data using following parameters:
            - 'criterion': 'gini'
            - 'max_depth': 3
            - 'min_samples_leaf': 2
            - 'N_estimators': 100
            - 'random_state': 2020
            - Note: You need to reproduce the process (code) from Q2, Q3, Q4 to obtain the correct result.
        """
        self.Q4() # remove this if you want to skip dependency and duplicate code

        self.rf = RandomForestClassifier(criterion='gini', max_depth=3, min_samples_leaf=2, n_estimators=100, random_state=2020)
        self.rf.fit(self.X_train, self.y_train)

        importances = pd.Series(self.rf.feature_importances_, index=self.X_train.columns)
        importances_sorted = importances.sort_values(ascending=False)
        return importances_sorted.index[1]


    def Q6(self):
        """
            6. (From step 10) What is the value of macro f1 (2 digits)?
            Predict the testing data set with confusion_matrix and classification_report,
            using scientific rounding (less than 0.5 dropped, more than 0.5 then increased)
            - Note: You need to reproduce the process (code) from Q2, Q3, Q4, Q5 to obtain the correct result.
        """
        self.Q5() # remove this if you want to skip dependency and duplicate code
        y_pred = self.rf.predict(self.X_test)
        
        f1_classes = f1_score(self.y_test, y_pred, average=None)

        return (float(round(f1_classes[0], 2)), float(round(f1_classes[1], 2)))


Run the code below to test that your code can work. Please **run one by one** as there are dependencies between question in the code to prevent duplicated codes. 

** **DO NOT SUBMIT THIS PART TO GRADER** **

In [6]:
hw = MushroomClassifier('mushroom2020_dataset.csv')

print(hw.Q1())
print(hw.Q2())
print(hw.Q3())
print(hw.Q4())
print(hw.Q5())
print(hw.Q6())

121
(5764, 12)
(3660, 2104)
((4611, 42), (1153, 42))
bruises_t
(0.98, 0.97)
